In [8]:
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 1. Load your JSON
with open('data/course-api-data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Build a DataFrame of code, title, description
rows = []
for code, info in data.items():
    api = info.get('api_data', {})
    rows.append({
        'courseCode': code,
        'title':       api.get('title',    info.get('title', '')),
        'description': api.get('description', '')
    })
df = pd.DataFrame(rows)

# 3. TF‑IDF → SVD pipeline
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['description'])

svd = TruncatedSVD(n_components=100, random_state=42)
embeddings = svd.fit_transform(X_tfidf)

# 4. Attach & save
df['embedding'] = embeddings.tolist()
df.to_pickle('data/course_embeddings.pkl')
np.save('data/course_embeddings.npy', embeddings)

# 5. (Optional) Quick peek at the table
df[['courseCode','title','description']].head()


,courseCode,title,description
0,ACTSC127,Introduction to Global Capital Markets and Fin...,This course introduces financial markets and i...
1,ACTSC221,Introductory Financial Mathematics (Non-Specia...,The theory of rates of interest and discount; ...
2,ACTSC231,Introductory Financial Mathematics,The theory of rates of interest and discount i...
3,ACTSC232,Life Contingencies 1,The future lifetime random variable: probabili...
4,ACTSC291,Global Capital Markets and Financial Analytics,This course offers an overview of global capit...


In [9]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pandas as pd

# Reload your course DataFrame
df = pd.read_pickle('data/course_embeddings.pkl')  # assumes you re-run the vectorization cell

# 1. Fit TF‑IDF & SVD on your course descriptions
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['description'])

svd = TruncatedSVD(n_components=100, random_state=42)
embeddings = svd.fit_transform(X_tfidf)

# 2. Overwrite embeddings & save models
import numpy as np
df['embedding'] = embeddings.tolist()
df.to_pickle('data/course_embeddings.pkl')
np.save('data/course_embeddings.npy', embeddings)

with open('data/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('data/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)

print("Models and embeddings saved.")


Models and embeddings saved.


In [13]:
# Method 1: Direct Cosine‑Similarity Ranking
import pickle, numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ── EDIT THIS: your free‑text search string ───────────────────────
search_query = "a course about dinosaurs, fossils and ancient civilizations"
# ───────────────────────────────────────────────────────────────────

# Load artifacts
df     = pd.read_pickle('data/course_embeddings.pkl')
emb    = np.load('data/course_embeddings.npy')
with open('data/tfidf_vectorizer.pkl','rb') as f: tfidf = pickle.load(f)
with open('data/svd_model.pkl','rb')         as f: svd   = pickle.load(f)

def recommend_cosine(query, top_k=10):
    q_vec = svd.transform(tfidf.transform([query]))               # (1,100)
    sims  = cosine_similarity(emb, q_vec).flatten()              # (N,)
    idxs  = sims.argsort()[::-1][:top_k]
    return df.iloc[idxs][['courseCode','title','description']].assign(similarity=sims[idxs])

# Run & display
top10 = recommend_cosine(search_query, top_k=10)
print(top10)


     courseCode                                        title  \
821     CLAS260                              Ancient Science   
3796     SCI266                              Ancient Science   
3173    PHIL260                              Ancient Science   
1978     GRK101                 Introductory Ancient Greek 1   
3614     RCS101                 Introductory Ancient Greek 1   
3615     RCS102                 Introductory Ancient Greek 2   
1979     GRK102                 Introductory Ancient Greek 2   
3793     SCI238                       Introductory Astronomy   
805     CLAS123             Classical Studies in Pop Culture   
2131    HIST317  History of Sexuality: The Pre-Modern Period   

                                            description  similarity  
821   The ancient Greeks developed scientific theori...    0.722057  
3796  The ancient Greeks developed scientific theori...    0.722057  
3173  The ancient Greeks developed scientific theori...    0.722057  
1978  This cour

In [18]:
# Method 2: Approximate Nearest‑Neighbor Search with FAISS
import pickle, faiss, numpy as np, pandas as pd

# ── EDIT THIS ───────────────────────────────────────────────────────
search_query = "analyze financial statements using Python and Excel"
# ───────────────────────────────────────────────────────────────────

# Load & normalize
df  = pd.read_pickle('data/course_embeddings.pkl')
emb = np.load('data/course_embeddings.npy').astype('float32')
faiss.normalize_L2(emb)

# Build or load index
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

with open('data/tfidf_vectorizer.pkl','rb') as f: tfidf = pickle.load(f)
with open('data/svd_model.pkl','rb')         as f: svd   = pickle.load(f)

def recommend_faiss(query, top_k=10):
    q = svd.transform(tfidf.transform([query])).astype('float32')
    faiss.normalize_L2(q)
    D, I = index.search(q, top_k)   # D=sim scores, I=indices
    return df.iloc[I[0]][['courseCode','title','description']].assign(similarity=D[0])

# Run & display
print(recommend_faiss(search_query, top_k=10))


     courseCode                                              title  \
1246    ECON206                                Money and Banking 1   
121      AFM422               Management of Financial Institutions   
98       AFM322                              Derivative Securities   
88       AFM272     Global Capital Markets and Financial Analytics   
4      ACTSC291     Global Capital Markets and Financial Analytics   
112      AFM373        Cases and Applications in Corporate Finance   
67       AFM101               Introduction to Financial Accounting   
89       AFM273          Financial Instruments and Capital Markets   
92       AFM276                       Financial Statement Analysis   
593      CFM101  Introduction to Financial Markets and Data Ana...   

                                            description  similarity  
1246  This course offers an overview of the function...    0.876117  
121   This course focuses on the economics of financ...    0.863707  
98    This course w

In [19]:
# Method 3: Diversity‑Aware Re‑Ranking (MMR)
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# ── EDIT THIS ───────────────────────────────────────────────────────
search_query = "analyze financial statements using Python and Excel"
# ───────────────────────────────────────────────────────────────────

# (Assumes tfidf, svd, emb, df already in scope)

def recommend_mmr(query, top_k=10, λ=0.7):
    q_vec      = svd.transform(tfidf.transform([query])).flatten()
    sims       = cosine_similarity(emb, q_vec.reshape(1,-1)).flatten()
    candidates = list(range(len(sims)))
    selected   = []

    for _ in range(top_k):
        if not selected:
            idx = int(np.argmax(sims))
        else:
            mmr_scores = []
            for i in candidates:
                rel = sims[i]
                red = max(cosine_similarity(emb[selected], emb[i].reshape(1,-1)).flatten())
                mmr_scores.append((i, λ*rel - (1-λ)*red))
            idx = max(mmr_scores, key=lambda x: x[1])[0]
        selected.append(idx)
        candidates.remove(idx)

    return df.iloc[selected][['courseCode','title','description']].assign(similarity=sims[selected])

# Run & display
print(recommend_mmr(search_query, top_k=10))


     courseCode                                              title  \
1246    ECON206                                Money and Banking 1   
112      AFM373        Cases and Applications in Corporate Finance   
98       AFM322                              Derivative Securities   
121      AFM422               Management of Financial Institutions   
4      ACTSC291     Global Capital Markets and Financial Analytics   
593      CFM101  Introduction to Financial Markets and Data Ana...   
887     COMM421                       Financial Statement Analysis   
67       AFM101               Introduction to Financial Accounting   
92       AFM276                       Financial Statement Analysis   
89       AFM273          Financial Instruments and Capital Markets   

                                            description  similarity  
1246  This course offers an overview of the function...    0.876117  
112   This course builds on the theory of financial ...    0.838005  
98    This course w

In [23]:
# Method 6: Graph‑Based Diffusion (Personalized PageRank)
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# ── EDIT THIS ───────────────────────────────────────────────────────
search_query = "analyze financial statements using Python and Excel"
# ───────────────────────────────────────────────────────────────────

# (Assumes tfidf, svd, emb, df already in scope)

def recommend_graph(query, top_k=10):
    q_vec = svd.transform(tfidf.transform([query]))               # (1,100)
    sims  = cosine_similarity(emb, q_vec).flatten()              # (N,)

    G = nx.DiGraph()
    G.add_node('query')
    for i, code in enumerate(df['courseCode']):
        G.add_edge('query', code, weight=float(sims[i]))

    pr = nx.pagerank(G, alpha=0.85, personalization={'query':1.0})
    ranked = sorted(
        ((c,sc) for c,sc in pr.items() if c!='query'),
        key=lambda x: x[1], reverse=True
    )[:top_k]
    codes  = [c for c,_ in ranked]
    scores = [s for _,s in ranked]
    return df[df['courseCode'].isin(codes)][['courseCode','title','description']].assign(score=scores)

# Run & display
print(recommend_graph(search_query, top_k=10))


     courseCode                                              title  \
4      ACTSC291     Global Capital Markets and Financial Analytics   
67       AFM101               Introduction to Financial Accounting   
88       AFM272     Global Capital Markets and Financial Analytics   
89       AFM273          Financial Instruments and Capital Markets   
92       AFM276                       Financial Statement Analysis   
98       AFM322                              Derivative Securities   
112      AFM373        Cases and Applications in Corporate Finance   
121      AFM422               Management of Financial Institutions   
593      CFM101  Introduction to Financial Markets and Data Ana...   
1246    ECON206                                Money and Banking 1   

                                            description     score  
4     This course offers an overview of global capit...  0.002609  
67    This course is an introduction to financial ac...  0.002572  
88    This course offers 

In [22]:
pip install networkx

Defaulting to user installation because normal site-packages is not writeable
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
Using cached networkx-3.5-py3-none-any.whl (2.0 MB)
Note: you may need to restart the kernel to use updated packages.
